**FASE 4c: FUSIÓN TEMPORAL, GRID SEARCH Y COMPARATIVA DE FPS**

Autor: Andoni Cabrera Fernández

Descripción: Este cuaderno actúa como el motor de diagnóstico del TFG aplicando el mapeo lógico-temporal (Late Fusion). En lugar de usar una red neuronal compleja, se aplican reglas fisiológicas basadas en la escala KSS:
1. PERCLOS (Porcentaje de Cierre Ocular) para la somnolencia crítica (Clase 10).
2. Tiempo de Bostezo Sostenido (en segundos) para la baja vigilancia (Clase 5).

Se evalúan los archivos `.pkl` a 5, 15 y 30 FPS. Mediante una búsqueda en cuadrícula (Grid Search), se determina qué combinación de umbral PERCLOS y tiempo de bostezo maximiza el F1-Score y si procesar a 30 FPS aporta alguna mejora sobre los 5 FPS.

In [5]:
# =========================================================
# FASE 4c: FUSIÓN TEMPORAL CNN, GRID SEARCH DUAL Y 1 FPS
# =========================================================
import pickle
import pandas as pd
import numpy as np
import time
from sklearn.metrics import accuracy_score, f1_score
from google.colab import drive

# ---------------------------------------------------------
# 1. CARGA DIRECTA DE DATOS (YA CREADO 1 FPS)
# ---------------------------------------------------------
drive.mount('/content/drive')
ruta_base = '/content/drive/MyDrive/TFG_Fatiga_Colab/PKL_CNN/'

print("Cargando inferencias de MobileNetV2 (TFLite) desde Drive...")
try:
    with open(f'{ruta_base}datos_TFLITE_1fps.pkl', 'rb') as f: df_1 = pd.DataFrame(pickle.load(f))
    with open(f'{ruta_base}datos_TFLITE_5fps.pkl', 'rb') as f: df_5 = pd.DataFrame(pickle.load(f))
    with open(f'{ruta_base}datos_TFLITE_15fps.pkl', 'rb') as f: df_15 = pd.DataFrame(pickle.load(f))
    with open(f'{ruta_base}datos_TFLITE_30fps.pkl', 'rb') as f: df_30 = pd.DataFrame(pickle.load(f))
except FileNotFoundError:
    print("Aviso: No se encontraron los archivos '_TFLITE'. Intentando nombres alternativos...")
    with open(f'{ruta_base}datos_1fps.pkl', 'rb') as f: df_1 = pd.DataFrame(pickle.load(f))
    with open(f'{ruta_base}datos_5fps.pkl', 'rb') as f: df_5 = pd.DataFrame(pickle.load(f))
    with open(f'{ruta_base}datos_15fps.pkl', 'rb') as f: df_15 = pd.DataFrame(pickle.load(f))
    with open(f'{ruta_base}datos_30fps.pkl', 'rb') as f: df_30 = pd.DataFrame(pickle.load(f))

print(f"Datos listos. Registros: 1FPS ({len(df_1)}), 5FPS ({len(df_5)}), 15FPS ({len(df_15)}), 30FPS ({len(df_30)})\n")

# ---------------------------------------------------------
# 2. MOTOR LÓGICO-TEMPORAL (LATE FUSION CNN)
# ---------------------------------------------------------
def evaluar_videos_cnn(df, fps, umbral_certeza_ojo, umbral_certeza_boca, umbral_perclos, umbral_segundos_bostezo):
    resultados = []
    for video_name, grupo in df.groupby('video'):
        total_frames = len(grupo)
        if total_frames == 0: continue
        clase_real = grupo['clase_real'].iloc[0]

        # 1. CÁLCULO DE PERCLOS NEURONAL
        frames_cerrados = (grupo['prob_ojo_cerrado'] > umbral_certeza_ojo).sum()
        perclos = frames_cerrados / total_frames

        # 2. CÁLCULO DE BOSTEZO SOSTENIDO
        es_bostezo = grupo['prob_bostezo'] > umbral_certeza_boca
        rachas = es_bostezo.groupby((~es_bostezo).cumsum()).sum()
        max_frames_consecutivos = rachas.max() if not rachas.empty else 0
        segundos_max_bostezo = max_frames_consecutivos / fps

        # 3. ÁRBOL DE DECISIÓN (Jerarquía KSS)
        if perclos >= umbral_perclos: clase_predicha = 10
        elif segundos_max_bostezo >= umbral_segundos_bostezo: clase_predicha = 5
        else: clase_predicha = 0

        resultados.append({'video': video_name, 'real': clase_real, 'pred': clase_predicha})
    return pd.DataFrame(resultados)

# ---------------------------------------------------------
# 3. GRID SEARCH DUAL (ZOOM EXTREMO)
# ---------------------------------------------------------
# Ajuste milimétrico final para acorralar el máximo rendimiento
espacio_certeza_ojo = [0.05, 0.10, 0.15, 0.20, 0.25]     # Bajamos casi al mínimo absoluto
espacio_certeza_boca = [0.80, 0.85, 0.90, 0.95, 0.99]    # Mantenemos la exigencia alta para la boca
espacio_perclos = [0.04, 0.05, 0.06, 0.07, 0.08, 0.10]   # Centrado en el nuevo límite superior
espacio_bostezo = [1.0, 1.2, 1.5, 2.0, 2.5, 3.0]         # Abrimos el techo hasta los 3.0 segundos

datasets = {'1 FPS': (df_1, 1), '5 FPS': (df_5, 5), '15 FPS': (df_15, 15), '30 FPS': (df_30, 30)}

mejores_modelos_f1 = {}
mejores_modelos_acc = {}

total_combs = len(espacio_certeza_ojo) * len(espacio_certeza_boca) * len(espacio_perclos) * len(espacio_bostezo)
print(f"🚀 Iniciando Grid Search de Zoom Extremo ({total_combs} combinaciones por resolución temporal)...")

for nombre_fps, (df_data, fps_val) in datasets.items():
    inicio_tiempo = time.time()
    resultados_grid = []

    for u_ojo in espacio_certeza_ojo:
        for u_boca in espacio_certeza_boca:
            for u_per in espacio_perclos:
                for u_bos in espacio_bostezo:
                    res_df = evaluar_videos_cnn(
                        df_data, fps=fps_val,
                        umbral_certeza_ojo=u_ojo, umbral_certeza_boca=u_boca,
                        umbral_perclos=u_per, umbral_segundos_bostezo=u_bos
                    )

                    acc = accuracy_score(res_df['real'], res_df['pred'])
                    f1 = f1_score(res_df['real'], res_df['pred'], average='macro')

                    resultados_grid.append({
                        'U_Ojo': u_ojo, 'U_Boca': u_boca,
                        'PERCLOS': u_per, 'Bostezo': u_bos,
                        'Accuracy': acc, 'F1_Score': f1
                    })

    tiempo_procesamiento = time.time() - inicio_tiempo
    df_resultados_fps = pd.DataFrame(resultados_grid)

    # Extraer el mejor por F1-Score
    df_top_f1 = df_resultados_fps.sort_values(by=['F1_Score', 'Accuracy'], ascending=[False, False]).head(1)
    mejores_modelos_f1[nombre_fps] = {
        'Top_Config': df_top_f1.iloc[0], 'Tiempo_Busqueda': tiempo_procesamiento
    }

    # Extraer el mejor por Accuracy
    df_top_acc = df_resultados_fps.sort_values(by=['Accuracy', 'F1_Score'], ascending=[False, False]).head(1)
    mejores_modelos_acc[nombre_fps] = {'Top_Config': df_top_acc.iloc[0]}

# ---------------------------------------------------------
# 4. IMPRESIÓN DE RESULTADOS DEFINITIVOS
# ---------------------------------------------------------
print("\n" + "="*80)
print("🎯 RESULTADOS RED NEURONAL DEFINITIVOS (PRIORIZANDO F1-SCORE)")
print("="*80)
for fps, datos in mejores_modelos_f1.items():
    top = datos['Top_Config']
    print(f"[{fps}] F1: {top['F1_Score']*100:.2f}% | Acc: {top['Accuracy']*100:.2f}% --> Certeza Ojo: {top['U_Ojo']} | Certeza Boca: {top['U_Boca']} | PERCLOS: {top['PERCLOS']} | Bostezo: {top['Bostezo']}s")

print("\n" + "="*80)
print("🛡️ RESULTADOS RED NEURONAL DEFINITIVOS (PRIORIZANDO ACCURACY)")
print("="*80)
for fps, datos in mejores_modelos_acc.items():
    top = datos['Top_Config']
    print(f"[{fps}] Acc: {top['Accuracy']*100:.2f}% | F1: {top['F1_Score']*100:.2f}% --> Certeza Ojo: {top['U_Ojo']} | Certeza Boca: {top['U_Boca']} | PERCLOS: {top['PERCLOS']} | Bostezo: {top['Bostezo']}s")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando inferencias de MobileNetV2 (TFLite) desde Drive...
Datos listos. Registros: 1FPS (101538), 5FPS (507333), 15FPS (1521826), 30FPS (3043585)

🚀 Iniciando Grid Search de Zoom Extremo (900 combinaciones por resolución temporal)...

🎯 RESULTADOS RED NEURONAL DEFINITIVOS (PRIORIZANDO F1-SCORE)
[1 FPS] F1: 25.51% | Acc: 45.56% --> Certeza Ojo: 0.25 | Certeza Boca: 0.95 | PERCLOS: 0.04 | Bostezo: 2.5s
[5 FPS] F1: 25.95% | Acc: 44.44% --> Certeza Ojo: 0.05 | Certeza Boca: 0.8 | PERCLOS: 0.1 | Bostezo: 2.5s
[15 FPS] F1: 26.61% | Acc: 46.67% --> Certeza Ojo: 0.1 | Certeza Boca: 0.85 | PERCLOS: 0.05 | Bostezo: 1.2s
[30 FPS] F1: 25.93% | Acc: 46.67% --> Certeza Ojo: 0.1 | Certeza Boca: 0.9 | PERCLOS: 0.05 | Bostezo: 1.0s

🛡️ RESULTADOS RED NEURONAL DEFINITIVOS (PRIORIZANDO ACCURACY)
[1 FPS] Acc: 46.67% | F1: 24.11% --> Certeza Ojo: 0.25 | Certeza Boca: 0.99 | PER

In [6]:
# =========================================================
# DIAGNÓSTICO DE SALUD DE LOS DATOS (SANITY CHECK)
# =========================================================
print("--- 1. RESUMEN ESTADÍSTICO DE LAS PREDICCIONES (5 FPS) ---")
print(df_5[['prob_ojo_cerrado', 'prob_bostezo']].describe())

print("\n--- 2. ¿HAY VALORES NULOS O VACÍOS? ---")
print(df_5.isna().sum())

print("\n--- 3. MUESTRA DE LAS PRIMERAS 10 FILAS ---")
print(df_5[['video', 'frame_idx', 'prob_ojo_cerrado', 'prob_bostezo', 'clase_real']].head(10))

print("\n--- 4. ¿CUÁNTAS PREDICCIONES ÚNICAS ESTÁ HACIENDO EL MOTOR AHORA MISMO? ---")
prueba_motor = evaluar_videos_cnn(df_5, 5, 0.5, 0.5, 0.15, 1.0)
print(prueba_motor['pred'].value_counts())

--- 1. RESUMEN ESTADÍSTICO DE LAS PREDICCIONES (5 FPS) ---
       prob_ojo_cerrado   prob_bostezo
count      5.073330e+05  507333.000000
mean       6.692289e-02       0.442627
std        1.869905e-01       0.307897
min        6.338146e-11       0.000174
25%        4.933355e-05       0.162813
50%        8.302834e-04       0.395007
75%        1.577663e-02       0.713708
max        9.997490e-01       0.999998

--- 2. ¿HAY VALORES NULOS O VACÍOS? ---
video               0
clase_real          0
frame_idx           0
prob_ojo_cerrado    0
prob_bostezo        0
dtype: int64

--- 3. MUESTRA DE LAS PRIMERAS 10 FILAS ---
      video  frame_idx  prob_ojo_cerrado  prob_bostezo  clase_real
0  01_0.mov          0          0.017055      0.230920           0
1  01_0.mov          6          0.410003      0.556795           0
2  01_0.mov         12          0.000608      0.842312           0
3  01_0.mov         18          0.000449      0.708355           0
4  01_0.mov         24          0.000387      